In [18]:
import torch
import torch.nn as nn
import math
import torch.optim as optim
import numpy as np

1.EMBEDDING

In [19]:
class TokenEmbedding(nn.Module):
  def __init__(self,size , d_model):
    super().__init__()
    self.embedding = nn.Embedding(size,d_model)
    self.scale = np.sqrt(d_model)
  def forward(self,x):
    return self.embedding(x)*self.scale

2.POSITIONAL ENCODING

In [20]:
class PositionalEncoding(nn.Module):
  def __init__(self, d_model , max_len=5000):
    super().__init__()
    pe = torch.zeros(max_len, d_model)
    pos = torch.arange(0,max_len).unsqueeze(1)
    div_term = torch.exp(torch.arange(0,d_model,2)*(-math.log(10000.0)/d_model))
    pe[:,0::2] = torch.sin(pos*div_term)
    pe[:,1::2] = torch.cos(pos*div_term)
    pe = pe.unsqueeze(0)
    self.register_buffer('pe',pe)
  def forward(self,x):
    return x+self.pe[:,:x.size(1)]


3.DOT PRODUCT ATTENTION

In [21]:
class DotProductAttention(nn.Module):
  def __init__(self, d_k):
    super().__init__()
    self.scale = np.sqrt(d_k)
  def forward(self, q,k,v, mask=None):
    scores = torch.matmul(q,k.transpose(-2,-1))/self.scale
    if mask is not None:
      scores = scores.masked_fill(mask==0, -1e9)
    attention = torch.softmax(scores , dim=-1)
    return torch.matmul(attention,v)

4.MULTIHEAD ATTENTION

In [22]:
class MultiHeadAttention(nn.Module):
  def __init__(self,d_model,n_heads):
    super().__init__()
    self.d_k = d_model // n_heads
    self.n_heads = n_heads
    self.q_linear = nn.Linear(d_model,d_model)
    self.k_linear = nn.Linear(d_model,d_model)
    self.v_linear = nn.Linear(d_model,d_model)
    self.out = nn.Linear(d_model,d_model)
  def forward(self,Q,K,V,mask = None):
      batch_size = Q.size(0)
      Q = self.q_linear(Q).view(batch_size,-1,self.n_heads,self.d_k).transpose(1,2)
      K = self.k_linear(K).view(batch_size,-1,self.n_heads,self.d_k).transpose(1,2)
      V = self.v_linear(V).view(batch_size,-1,self.n_heads,self.d_k).transpose(1,2)
      attention_out = DotProductAttention(self.d_k)(Q,K,V,mask)
      attention_out = attention_out.transpose(1,2).contiguous().view(batch_size,-1,self.n_heads*self.d_k)
      return self.out(attention_out)


5.LAYER NORMALIZATION

In [23]:
class LayerNormalization(nn.Module):
  def __init__(self,d_model, eps = 1e-5):
    super().__init__()
    self.eps = eps
    self.gama = nn.Parameter(torch.ones(d_model))
    self.beta = nn.Parameter(torch.zeros(d_model))
  def forward(self,x):
    mean = x.mean(dim=-1,keepdim=True)
    var = ((x-mean)**2).mean(dim = -1,keepdim=True)
    x_norm = (x-mean)/torch.sqrt(var + self.eps)
    return self.gama*x_norm+self.beta


6.POSITIONAL FORWARD FEEDING

In [24]:
class PositionalForwardFeeding(nn.Module):
  def __init__(self,d_model,d_ff,dropout=0.1):
    super().__init__()
    self.linear1 = nn.Linear(d_model,d_ff) #expand
    self.linear2 = nn.Linear(d_ff,d_model) #compress
    self.relu = nn.ReLU()
    self.dropout = nn.Dropout(dropout)

  def forward(self,x):
      return self.linear2(self.dropout(self.relu(self.linear1(x))))

In [25]:
class ResidualConnection(nn.Module):
  def __init__(self,dropout,d_model):
    super().__init__()
    self.dropout = nn.Dropout(dropout)
    self.norm = LayerNormalization(d_model)
  def forward(self,x,sublayer):
    return x + self.dropout(sublayer(self.norm(x)))

In [26]:
class EncoderLayer(nn.Module):
  def __init__(self,d_model,n_heads,d_ff,dropout = 0.1):
    super().__init__()
    self.self_attn = MultiHeadAttention(d_model,n_heads)
    self.ffn = PositionalForwardFeeding(d_model,d_ff)
    self.residual1 = ResidualConnection(dropout,d_model)
    self.residual2 = ResidualConnection(dropout,d_model)

  def forward(self,x,mask=None):
      x=self.residual1(x,
                       lambda x : self.self_attn(x,x,x,mask))
      x = self.residual2(x,self.ffn)
      return x

In [27]:
class Encoder(nn.Module):
  def __init__(self,d_model,vocab_size,n_heads,n_layers,d_ff,max_len,dropout=0.1):
    super().__init__()
    self.embedding = TokenEmbedding(vocab_size,d_model)
    self.pe = PositionalEncoding(d_model,max_len)
    self.layers = nn.ModuleList([
        EncoderLayer(d_model,n_heads,d_ff,dropout)
        for _ in range(n_layers)
      ])
  def forward(self,x,mask=None):
      x = self.embedding(x)
      x = self.pe(x)
      for layer in self.layers:
        x=layer(x,mask)
      return x


In [28]:
batch_size = 2
seq_len = 10
vocab_size = 10000
x = torch.randint(0, vocab_size, (batch_size, seq_len))

encoder = Encoder(
    vocab_size=vocab_size,
    d_model=512,
    n_heads=8,
    d_ff=2048,
    n_layers=6,
    max_len=100,
    dropout=0.1
)

out = encoder(x)
print(out.shape)

torch.Size([2, 10, 512])


In [29]:
x = torch.randint(0, vocab_size, (2, 10))
out = encoder(x)
print(out.shape)

torch.Size([2, 10, 512])


In [30]:
d_model = 512
max_len = 50

vocab = {
    "<pad>": 0,
    "i": 1,
    "love": 2,
    "transformers": 3
}

sentence = "I love transformers"
tokens = [vocab[word.lower()] for word in sentence.split()]
x = torch.tensor(tokens).unsqueeze(0)


encoder = Encoder(
    vocab_size=len(vocab),
    d_model=512,
    n_heads=8,
    d_ff=2048,
    n_layers=6,
    max_len=100
)
contextual_embeddings = encoder(x)

print(contextual_embeddings.shape)

torch.Size([1, 3, 512])


In [31]:
class DecoderLayer(nn.Module):
  def __init__(self, d_model, n_heads,dropout=0.1):
    super().__init__()

    self.self_attn = MultiHeadAttention(d_model, n_heads)
    self.coss_attn = MultiHeadAttention(d_model, n_heads)
    self.feed_forward = PositionalForwardFeeding(d_model)
    self.residual1 = ResidualConnection(dropout,d_model)
    self.residual2 = ResidualConnection(dropout,d_model)
    self.residual3 = ResidualConnection(dropout,d_model)
  def forward(self,x,encoder_output, src_mask=None,tgt_mask=None):
    x = self.residual1(x,lambda x : self.self_attn(x,x,x,tgt_mask))
    x = self.residual2(x,lambda x : self.cross_attn(x,encoder_output,encoder_output,src_mask))
    x = self.residual3(x,self.feed_forward)
    return x

In [32]:
class Decoder(nn.Module):
  def __init__(self,d_model,vocab_size,n_heads,n_layers,d_ff,max_len,dropout=0.1):
    super().__init__()
    self.embedding = TokenEmbedding(vocab_size,d_model)
    self.pe = PositionalEncoding(d_model,max_len)
    self.layers = nn.ModuleList([
        DecoderLayer(d_model,n_heads,d_ff,dropout)
        for _ in range(n_layers)
    ])

  def forward(self,x,encoder_out,src_mask=None,tgt_mask = None):
    embedding = self.embedding(x)
    pe = self.pe(x)
    for layer in self.layers:
      x=layer(x,encoder_out,src_mask,tgt_mask)
    return x;

In [33]:
class Projection(nn.Module):
  def __init__(self , d_model, vocab_size):
    super().__init__()
    self.projection = nn.Linear(d_model,vocab_size)
  def forward(self,x):
    return torch.log_softmax(self.projection(x),dim = -1)

In [34]:
class Transformer(nn.Module):
  def __init__(self, d_model , vocab_size ,n_layers,n_heads,d_ff,dropout=0.1):
    super().__init__()
    self.encoder = Encoder(d_model,vocab_size,n_heads,n_layers,d_ff,max_len, dropout)
    self.decoder = Decoder(d_model, vocab_size,n_heads,n_layers,d_ff,max_len,dropout)
    self.inpEmd = TokenEmbedding(vocab_size,d_model)
    self.outEmd = TokenEmbedding(vocab_size,d_model)
    self.inpPos = PositionalEncoding(d_model,max_len)
    self.outPos = PositionalEncoding(d_model,max_len)
    self.projection = Projection(d_model,vocab_size)
  def encode(self,x,src_mask):
    x = self.inpEmd(x)
    x = self.inpPos(x)
    return self.encoder(x,src_mask)
  def decode(self,x,encoder_out,src_mask,tgt_mask):
    x = self.outEmd(x)
    x = self.outPos(x)
    return self.encoder(x,encoder_out,src_mask,tgt_mask)
  def project(self,x):
    return self.projection(x)

In [35]:
def build_transformer(vocab_size,tgt_vocab_size,src_len , tgt_len , d_model = 512 , n_heads=6 , h = 8, dropout = 0.1, d_ff = 2048):
  src_emb = TokenEmbedding(vocab_size,d_model)
  tgt_emb = TokenEmbedding(tgt_vocab_size,d_model)
  src_pos = PositionalEncoding(d_model,src_len)
  tgt_pos = PositionalEncoding(d_model,tgt_len)
  encoder_blocks = []
  for _ in range(n_heads):
    encoder_self_attn = MultiHeadAttention(d_model,h)
    encoder_ffn = PositionalForwardFeeding(d_model,d_ff,dropout)
    encoder_block = EncoderLayer(encoder_self_attn,encoder_ffn, dropout)
    encoder_blocks.append(encoder_block)
  decoder_blocks = []
  for _ in range(n_heads):
    decoder_self_attn = MultiHeadAttention(d_model,h)
    decoder_cross_attn = MultiHeadAttention(d_model,h)
    decoder_ffn = PositionalForwardFeeding(d_model,d_ff,dropout)
    decoder_block = DecoderLayer(decoder_self_attn,decoder_cross_attn,decoder_ffn,dropout)
    decoder_blocks.append(decoder_block)
  encoder = Encoder(nn.ModuleList(encoder_blocks))
  decoder = Decoder(nn.ModuleList(decoder_blocks))

  projection_layer = Projection(d_model,tgt_vocab_size)
  transformer = Transformer(encoder,decoder,src_emb,tgt_emb,src_pos,tgt_pos,projection_layer)
  for p in transformer.parameters():
    if p.dim() > 1:
      nn.init.xavier_uniform_(p)
  return transformer